# 🍽️ T17: Meal Calorie Helper — Agentic AI Demo

**CSE 476 — Agentic AI and Intelligent Automation | CA1 Assignment**

---

## What This Notebook Demonstrates

This notebook demonstrates a **real AI agent** that:
- Maintains a **plan-act loop** (decide → call tool → read result → decide again)
- Calls **real Python tool functions** (not faked responses)
- Uses **session memory** to remember meals and budgets across turns
- Provides a **macro split** (protein / carbs / fat percentages)
- Makes **decisions** based on tool results

### Architecture

```
User Message
     │
     ▼
  MealCaloricAgent.chat()
     │
     ├── detect_intent()           ← PLAN phase
     │
     ├── call tools (tool 1 → 2 → ...) ← ACT phase
     │
     ├── read tool results         ← OBSERVE phase
     │
     └── compose final answer      ← RESPOND phase
```

**Files:**
- `food_database.py` — built-in nutrition data for 40+ foods
- `memory.py` — session state (budget, meals, totals, macros)
- `tools.py` — 5 tool functions the agent calls
- `agent.py` — plan-act loop + intent detection + response composer

⚠️ *All nutritional values are estimated. Actual values vary by brand, preparation, and serving size. For informational purposes only.*

---
## Setup

In [1]:
# Add the project folder to the Python path so imports work
import sys, os

# Make sure we're running from the T17_Meal_Calorie_Helper directory
notebook_dir = os.path.dirname(os.path.abspath('__file__'))
project_dir = os.path.join(notebook_dir, 'T17_Meal_Calorie_Helper')

# If notebook is inside the project folder already, use current dir
if os.path.exists(os.path.join(notebook_dir, 'agent.py')):
    project_dir = notebook_dir

if project_dir not in sys.path:
    sys.path.insert(0, project_dir)

print(f'Project directory: {project_dir}')
print('Imports ready ✅')

Project directory: F:\7th sem\Agentic AI CSE 476\ca1 agent\T17_Meal_Calorie_Helper
Imports ready ✅


In [2]:
# Import and reset the agent for a clean session
import importlib

import tools
import agent as agent_module
from agent import MealCaloricAgent

# Reset session memory (important if re-running cells)
tools.reset_session()

# Create the agent with verbose=True to show plan-act traces
agent = MealCaloricAgent(verbose=True)

print('Agent initialised ✅')
print('Session memory reset ✅')
print('verbose=True — you will see [PLAN] [ACT] [TOOL RESULT] [OBSERVE] traces')

Agent initialised ✅
Session memory reset ✅
verbose=True — you will see [PLAN] [ACT] [TOOL RESULT] [OBSERVE] traces


---
## 📋 Scenario 1 — Add Meals and Track Calories

**Goal:** Set a daily budget, log multiple meals, and observe how the agent:
1. Detects intent
2. Calls `lookup_calories()` tool first
3. Reads the result
4. Calls `add_meal()` tool
5. Reads memory
6. Produces a final response with macros

In [3]:
# ── Turn 1: Set the daily calorie budget ──────────────────────────────────
response = agent.chat("My calorie budget is 2200 kcal")


════════════════════════════════════════════════════════════
  👤 USER (Turn 1): My calorie budget is 2200 kcal
════════════════════════════════════════════════════════════
  [PLAN] Intent detected → SET_BUDGET

───────────────────────────────────────────────────────
  🤖 AGENT — Setting daily calorie budget
───────────────────────────────────────────────────────
  [ACT] Calling set_budget(2200.0)
  [TOOL RESULT] Status = set, budget = 2200.0 kcal

  🤖 RESPONSE:
✅ Daily calorie budget set to 2200 kcal.
   Start logging your meals and I'll track calories and macros for you.

⚠️  All nutritional values are estimated. Actual values vary by brand, preparation, and serving size. This tool is for informational purposes only.


In [4]:
# ── Turn 2: Add 3 boiled eggs ─────────────────────────────────────────────
#
# Plan-Act trace you will see:
#   [PLAN]        Intent → ADD_MEAL
#   [ACT]         lookup_calories('3 boiled eggs')
#   [TOOL RESULT] Found 'egg' — 234 kcal
#   [ACT]         add_meal('3 boiled eggs')
#   [TOOL RESULT] Daily total = 234 kcal
#   [MEMORY]      Consumed = 234 / 2200 kcal
#
response = agent.chat("Add 3 boiled eggs")


════════════════════════════════════════════════════════════
  👤 USER (Turn 2): Add 3 boiled eggs
════════════════════════════════════════════════════════════
  [PLAN] Intent detected → ADD_MEAL

───────────────────────────────────────────────────────
  🤖 AGENT — Logging a meal
───────────────────────────────────────────────────────
  [ACT] Calling lookup_calories('3 boiled eggs')
  [TOOL RESULT] Status = found
  [OBSERVE] Found 'egg' — 234.0 kcal, P:18.0g C:1.8g F:15.9g
  [ACT] Calling add_meal('3 boiled eggs')
  [TOOL RESULT] Status = added, Daily total = 234.0 kcal
  [MEMORY] Consumed = 234.0 / 2200.0 kcal | Meals logged = 1

  🤖 RESPONSE:
✅ Added: Egg (3 pieces)
  🔥 Calories: 234.0 kcal
  🥩 Protein : 18.0 g
  🍞 Carbs   : 1.8 g
  🧈 Fat     : 15.9 g
  📅 Today: 234.0 / 2200.0 kcal
  ⏳ Remaining: 1966.0 kcal

  📊 Macro Split (approximate):
     Protein : 18.0 g  → 32.4% of macro calories
     Carbs   : 1.8 g  → 3.2% of macro calories
     Fat     : 15.9 g  → 64.4% of macro calories
  

In [5]:
# ── Turn 3: Add 200 g chicken breast ─────────────────────────────────────
response = agent.chat("Add 200 g chicken breast")


════════════════════════════════════════════════════════════
  👤 USER (Turn 3): Add 200 g chicken breast
════════════════════════════════════════════════════════════
  [PLAN] Intent detected → ADD_MEAL

───────────────────────────────────────────────────────
  🤖 AGENT — Logging a meal
───────────────────────────────────────────────────────
  [ACT] Calling lookup_calories('200 g chicken breast')
  [TOOL RESULT] Status = found
  [OBSERVE] Found 'chicken breast' — 330.0 kcal, P:62.0g C:0.0g F:7.2g
  [ACT] Calling add_meal('200 g chicken breast')
  [TOOL RESULT] Status = added, Daily total = 564.0 kcal
  [MEMORY] Consumed = 564.0 / 2200.0 kcal | Meals logged = 2

  🤖 RESPONSE:
✅ Added: Chicken Breast (200.0 g)
  🔥 Calories: 330.0 kcal
  🥩 Protein : 62.0 g
  🍞 Carbs   : 0.0 g
  🧈 Fat     : 7.2 g
  📅 Today: 564.0 / 2200.0 kcal
  ⏳ Remaining: 1636.0 kcal

  📊 Macro Split (approximate):
     Protein : 80.0 g  → 59.8% of macro calories
     Carbs   : 1.8 g  → 1.3% of macro calories
     Fat   

In [6]:
# ── Turn 4: Add 2 slices of bread ────────────────────────────────────────
response = agent.chat("Add 2 slices of bread")


════════════════════════════════════════════════════════════
  👤 USER (Turn 4): Add 2 slices of bread
════════════════════════════════════════════════════════════
  [PLAN] Intent detected → ADD_MEAL

───────────────────────────────────────────────────────
  🤖 AGENT — Logging a meal
───────────────────────────────────────────────────────
  [ACT] Calling lookup_calories('2 slices of bread')
  [TOOL RESULT] Status = found
  [OBSERVE] Found 'bread' — 158.0 kcal, P:5.4g C:30.0g F:2.0g
  [ACT] Calling add_meal('2 slices of bread')
  [TOOL RESULT] Status = added, Daily total = 722.0 kcal
  [MEMORY] Consumed = 722.0 / 2200.0 kcal | Meals logged = 3

  🤖 RESPONSE:
✅ Added: Bread (2.0 slice)
  🔥 Calories: 158.0 kcal
  🥩 Protein : 5.4 g
  🍞 Carbs   : 30.0 g
  🧈 Fat     : 2.0 g
  📅 Today: 722.0 / 2200.0 kcal
  ⏳ Remaining: 1478.0 kcal

  📊 Macro Split (approximate):
     Protein : 85.4 g  → 49.2% of macro calories
     Carbs   : 31.8 g  → 18.3% of macro calories
     Fat     : 25.1 g  → 32.5% of 

In [7]:
# ── Turn 5: Show today's summary with macro split ─────────────────────────
response = agent.chat("Show today's summary")


════════════════════════════════════════════════════════════
  👤 USER (Turn 5): Show today's summary
════════════════════════════════════════════════════════════
  [PLAN] Intent detected → SHOW_SUMMARY

───────────────────────────────────────────────────────
  🤖 AGENT — Retrieving daily summary
───────────────────────────────────────────────────────
  [ACT] Calling get_daily_summary()
  [TOOL RESULT] Budget=2200.0, Consumed=722.0 kcal, Meals=3

  🤖 RESPONSE:
📋 Today's Meal Log:
   1. Egg (3 pieces) — 234.0 kcal | P:18.0g C:1.8g F:15.9g
   2. Chicken Breast (200.0 g) — 330.0 kcal | P:62.0g C:0.0g F:7.2g
   3. Bread (2.0 slice) — 158.0 kcal | P:5.4g C:30.0g F:2.0g

  [██████░░░░░░░░░░░░░░] 32.8%
  Consumed : 722.0 / 2200.0 kcal
  Remaining: 1478.0 kcal ✅

  📊 Macro Split (approximate):
     Protein : 85.4 g  → 49.2% of macro calories
     Carbs   : 31.8 g  → 18.3% of macro calories
     Fat     : 25.1 g  → 32.5% of macro calories
  ⚡ Total from macros: ~694.7 kcal
  (Protein×4 + Carbs×4

In [8]:
# ── Inspect raw memory state (showing programmatic storage) ──────────────
print("\n📦 RAW MEMORY STATE (proof that memory is programmatic, not LLM text):")
print("─" * 60)
import json
mem = tools.get_memory()
summary = mem.summary()
print(json.dumps(summary, indent=2))


📦 RAW MEMORY STATE (proof that memory is programmatic, not LLM text):
────────────────────────────────────────────────────────────
{
  "daily_budget": 2200.0,
  "total_calories": 722.0,
  "remaining_calories": 1478.0,
  "total_protein": 85.4,
  "total_carbs": 31.8,
  "total_fat": 25.1,
  "meal_count": 3,
  "meals": [
    {
      "food": "egg",
      "quantity": "3 pieces",
      "calories": 234.0,
      "protein": 18.0,
      "carbs": 1.8,
      "fat": 15.9
    },
    {
      "food": "chicken breast",
      "quantity": "200.0 g",
      "calories": 330.0,
      "protein": 62.0,
      "carbs": 0.0,
      "fat": 7.2
    },
    {
      "food": "bread",
      "quantity": "2.0 slice",
      "calories": 158.0,
      "protein": 5.4,
      "carbs": 30.0,
      "fat": 2.0
    }
  ]
}


---
## 🍌 Scenario 2 — Snack Decision (Snack FITS Budget)

**Goal:** Ask whether a banana fits the remaining calorie budget.

**Plan-Act trace:**
```
User: 'Can I eat a banana?'
  → [PLAN]   Intent = CHECK_SNACK
  → [MEMORY] Read current consumed kcal
  → [ACT]    lookup_calories('banana')
  → [OBSERVE] Snack = 105 kcal
  → [ACT]    check_snack_fit('banana')
  → [OBSERVE] Remaining budget vs snack calories
  → [DECIDE] 105 kcal < remaining → FITS ✅
  → [FINAL]  Yes, it fits. Here's why.
```

In [9]:
# ── Scenario 2: Can I eat a banana? ──────────────────────────────────────
# (After Scenario 1, we should still have plenty of budget remaining)
response = agent.chat("Can I eat a banana?")


════════════════════════════════════════════════════════════
  👤 USER (Turn 6): Can I eat a banana?
════════════════════════════════════════════════════════════
  [PLAN] Intent detected → CHECK_SNACK

───────────────────────────────────────────────────────
  🤖 AGENT — Checking snack feasibility
───────────────────────────────────────────────────────
  [MEMORY] Budget = 2200.0, Consumed = 722.0 kcal
  [ACT] Calling lookup_calories('banana')
  [TOOL RESULT] Status = found
  [OBSERVE] Snack = 'banana' → 105.0 kcal
  [ACT] Calling check_snack_fit('banana')
  [TOOL RESULT] Status = fits
  [OBSERVE] Remaining budget = 1478.0 kcal | Snack = 105.0 kcal | Surplus/deficit = 1373.0 kcal
  [DECIDE] FITS ✅

  🤖 RESPONSE:
✅ YES — Banana (1 piece) is approximately 105.0 kcal. You have 1478.0 kcal remaining, so it fits your budget with 1373.0 kcal to spare.

  Snack nutrition:
  🥩 Protein: 1.3 g | 🍞 Carbs: 27.0 g | 🧈 Fat: 0.4 g

  Budget overview:
  [██████░░░░░░░░░░░░░░] 32.8%
  Consumed : 722.0 / 2

In [10]:
# ── Also add the banana so it's logged ───────────────────────────────────
response = agent.chat("Add a banana")


════════════════════════════════════════════════════════════
  👤 USER (Turn 7): Add a banana
════════════════════════════════════════════════════════════
  [PLAN] Intent detected → ADD_MEAL

───────────────────────────────────────────────────────
  🤖 AGENT — Logging a meal
───────────────────────────────────────────────────────
  [ACT] Calling lookup_calories('a banana')
  [TOOL RESULT] Status = found
  [OBSERVE] Found 'banana' — 105.0 kcal, P:1.3g C:27.0g F:0.4g
  [ACT] Calling add_meal('a banana')
  [TOOL RESULT] Status = added, Daily total = 827.0 kcal
  [MEMORY] Consumed = 827.0 / 2200.0 kcal | Meals logged = 4

  🤖 RESPONSE:
✅ Added: Banana (1 piece)
  🔥 Calories: 105.0 kcal
  🥩 Protein : 1.3 g
  🍞 Carbs   : 27.0 g
  🧈 Fat     : 0.4 g
  📅 Today: 827.0 / 2200.0 kcal
  ⏳ Remaining: 1373.0 kcal

  📊 Macro Split (approximate):
     Protein : 86.7 g  → 42.7% of macro calories
     Carbs   : 58.8 g  → 29.0% of macro calories
     Fat     : 25.5 g  → 28.3% of macro calories
  ⚡ Total fr

---
## ❌ Scenario 3 — Snack That Does NOT Fit Budget

**Goal:** Create a situation where the remaining budget is smaller than the snack.

We'll:
1. Reset and set a **tight 400 kcal budget** (diet day scenario)
2. Log meals that consume ~350 kcal
3. Ask if we can eat soya chunks (345 kcal / 100 g)

The agent should say ❌ NO and explain why.

In [11]:
# ── Setup for Scenario 3: Fresh session with tight budget ─────────────────
tools.reset_session()
agent._turn = 0   # reset turn counter for cleaner display

print("Session reset ✅ — starting Scenario 3")

Session reset ✅ — starting Scenario 3


In [12]:
# ── Set a tight budget ───────────────────────────────────────────────────
response = agent.chat("My calorie budget is 400 kcal")


════════════════════════════════════════════════════════════
  👤 USER (Turn 1): My calorie budget is 400 kcal
════════════════════════════════════════════════════════════
  [PLAN] Intent detected → SET_BUDGET

───────────────────────────────────────────────────────
  🤖 AGENT — Setting daily calorie budget
───────────────────────────────────────────────────────
  [ACT] Calling set_budget(400.0)
  [TOOL RESULT] Status = set, budget = 400.0 kcal

  🤖 RESPONSE:
✅ Daily calorie budget set to 400 kcal.
   Start logging your meals and I'll track calories and macros for you.

⚠️  All nutritional values are estimated. Actual values vary by brand, preparation, and serving size. This tool is for informational purposes only.


In [13]:
# ── Log two roti (~104 kcal each) → 208 kcal consumed ───────────────────
response = agent.chat("Add 2 roti")


════════════════════════════════════════════════════════════
  👤 USER (Turn 2): Add 2 roti
════════════════════════════════════════════════════════════
  [PLAN] Intent detected → ADD_MEAL

───────────────────────────────────────────────────────
  🤖 AGENT — Logging a meal
───────────────────────────────────────────────────────
  [ACT] Calling lookup_calories('2 roti')
  [TOOL RESULT] Status = found
  [OBSERVE] Found 'roti' — 208.0 kcal, P:6.2g C:36.0g F:5.4g
  [ACT] Calling add_meal('2 roti')
  [TOOL RESULT] Status = added, Daily total = 208.0 kcal
  [MEMORY] Consumed = 208.0 / 400.0 kcal | Meals logged = 1

  🤖 RESPONSE:
✅ Added: Roti (2 pieces)
  🔥 Calories: 208.0 kcal
  🥩 Protein : 6.2 g
  🍞 Carbs   : 36.0 g
  🧈 Fat     : 5.4 g
  📅 Today: 208.0 / 400.0 kcal
  ⏳ Remaining: 192.0 kcal

  📊 Macro Split (approximate):
     Protein : 6.2 g  → 11.4% of macro calories
     Carbs   : 36.0 g  → 66.2% of macro calories
     Fat     : 5.4 g  → 22.4% of macro calories
  ⚡ Total from macros: ~21

In [14]:
# ── Log 100g curd (~98 kcal) → now at ~306 kcal consumed ─────────────────
response = agent.chat("Add 100 g curd")


════════════════════════════════════════════════════════════
  👤 USER (Turn 3): Add 100 g curd
════════════════════════════════════════════════════════════
  [PLAN] Intent detected → ADD_MEAL

───────────────────────────────────────────────────────
  🤖 AGENT — Logging a meal
───────────────────────────────────────────────────────
  [ACT] Calling lookup_calories('100 g curd')
  [TOOL RESULT] Status = found
  [OBSERVE] Found 'curd' — 98.0 kcal, P:11.0g C:3.4g F:4.3g
  [ACT] Calling add_meal('100 g curd')
  [TOOL RESULT] Status = added, Daily total = 306.0 kcal
  [MEMORY] Consumed = 306.0 / 400.0 kcal | Meals logged = 2

  🤖 RESPONSE:
✅ Added: Curd (100.0 g)
  🔥 Calories: 98.0 kcal
  🥩 Protein : 11.0 g
  🍞 Carbs   : 3.4 g
  🧈 Fat     : 4.3 g
  📅 Today: 306.0 / 400.0 kcal
  ⏳ Remaining: 94.0 kcal

  📊 Macro Split (approximate):
     Protein : 17.2 g  → 21.9% of macro calories
     Carbs   : 39.4 g  → 50.2% of macro calories
     Fat     : 9.7 g  → 27.8% of macro calories
  ⚡ Total from ma

In [15]:
# ── Now ask if we can eat 100g soya chunks (345 kcal) ─────────────────────
#
# Budget = 400 kcal
# Consumed ≈ 306 kcal
# Remaining ≈ 94 kcal
# Soya chunks (100g) = 345 kcal  →  DOES NOT FIT ❌
#
# Plan-Act trace:
#   [PLAN]   Intent = CHECK_SNACK
#   [MEMORY] Budget=400, Consumed≈306, Remaining≈94
#   [ACT]    lookup_calories('100 g soya chunks')
#   [OBSERVE] Snack = 345 kcal
#   [ACT]    check_snack_fit('100 g soya chunks')
#   [OBSERVE] 94 - 345 = -251 → DOES NOT FIT
#   [DECIDE] ❌ Exceeds budget by 251 kcal
#
response = agent.chat("Can I eat 100 g soya chunks?")


════════════════════════════════════════════════════════════
  👤 USER (Turn 4): Can I eat 100 g soya chunks?
════════════════════════════════════════════════════════════
  [PLAN] Intent detected → CHECK_SNACK

───────────────────────────────────────────────────────
  🤖 AGENT — Checking snack feasibility
───────────────────────────────────────────────────────
  [MEMORY] Budget = 400.0, Consumed = 306.0 kcal
  [ACT] Calling lookup_calories('100 g soya chunks')
  [TOOL RESULT] Status = found
  [OBSERVE] Snack = 'soya chunks' → 345.0 kcal
  [ACT] Calling check_snack_fit('100 g soya chunks')
  [TOOL RESULT] Status = does_not_fit
  [OBSERVE] Remaining budget = 94.0 kcal | Snack = 345.0 kcal | Surplus/deficit = -251.0 kcal
  [DECIDE] DOES NOT FIT ❌

  🤖 RESPONSE:
❌ NO — Soya Chunks (100.0 g) is approximately 345.0 kcal. You only have 94.0 kcal remaining, so you would exceed your budget by 251.0 kcal.

  Snack nutrition:
  🥩 Protein: 52.0 g | 🍞 Carbs: 33.0 g | 🧈 Fat: 0.5 g

  Budget overview:

In [16]:
# ── Show final summary ───────────────────────────────────────────────────
response = agent.chat("Show today's summary")


════════════════════════════════════════════════════════════
  👤 USER (Turn 5): Show today's summary
════════════════════════════════════════════════════════════
  [PLAN] Intent detected → SHOW_SUMMARY

───────────────────────────────────────────────────────
  🤖 AGENT — Retrieving daily summary
───────────────────────────────────────────────────────
  [ACT] Calling get_daily_summary()
  [TOOL RESULT] Budget=400.0, Consumed=306.0 kcal, Meals=2

  🤖 RESPONSE:
📋 Today's Meal Log:
   1. Roti (2 pieces) — 208.0 kcal | P:6.2g C:36.0g F:5.4g
   2. Curd (100.0 g) — 98.0 kcal | P:11.0g C:3.4g F:4.3g

  [███████████████░░░░░] 76.5%
  Consumed : 306.0 / 400.0 kcal
  Remaining: 94.0 kcal ✅

  📊 Macro Split (approximate):
     Protein : 17.2 g  → 21.9% of macro calories
     Carbs   : 39.4 g  → 50.2% of macro calories
     Fat     : 9.7 g  → 27.8% of macro calories
  ⚡ Total from macros: ~313.7 kcal
  (Protein×4 + Carbs×4 + Fat×9 — approximate; for informational use only)

⚠️  All nutritional valu

---
## 🔍 Scenario 4 — Unknown Food (Honest Failure Handling)

When a food is not in the database, the agent should:
- Tell the user the food is unavailable
- NOT invent nutrition values
- Suggest trying a different name

In [17]:
# ── Unknown food test ────────────────────────────────────────────────────
response = agent.chat("Add a plate of biryani")


════════════════════════════════════════════════════════════
  👤 USER (Turn 6): Add a plate of biryani
════════════════════════════════════════════════════════════
  [PLAN] Intent detected → ADD_MEAL

───────────────────────────────────────────────────────
  🤖 AGENT — Logging a meal
───────────────────────────────────────────────────────
  [ACT] Calling lookup_calories('a plate of biryani')
  [TOOL RESULT] Status = not_found

  🤖 RESPONSE:
❌ Food not found: 'a plate of biryani' is not in the food database. Please try a different food name or check the supported list. Do not guess nutritional values for unknown foods.
   Tip: Try a more common food name from the supported list.


In [18]:
# ── Unknown snack check ──────────────────────────────────────────────────
response = agent.chat("Can I eat a samosa?")


════════════════════════════════════════════════════════════
  👤 USER (Turn 7): Can I eat a samosa?
════════════════════════════════════════════════════════════
  [PLAN] Intent detected → CHECK_SNACK

───────────────────────────────────────────────────────
  🤖 AGENT — Checking snack feasibility
───────────────────────────────────────────────────────
  [MEMORY] Budget = 400.0, Consumed = 306.0 kcal
  [ACT] Calling lookup_calories('samosa')
  [TOOL RESULT] Status = not_found

  🤖 RESPONSE:
❌ Food not found: 'samosa' is not in the food database. Please try a different food name or check the supported list. Do not guess nutritional values for unknown foods.
   Please try a common food name.


---
## 🔧 Direct Tool Call Demo

This section shows the **raw tool functions** the agent uses.
It proves that the tools return structured data which the agent then reasons about.

In [19]:
# ── Direct tool call: lookup_calories ────────────────────────────────────
import json
import tools

print("=" * 55)
print("TOOL CALL: lookup_calories('3 boiled eggs')")
print("=" * 55)
result = tools.lookup_calories("3 boiled eggs")
print(json.dumps(result, indent=2))

TOOL CALL: lookup_calories('3 boiled eggs')
{
  "food": "egg",
  "serving_size": "1 large egg (~50 g)",
  "quantity_str": "3 pieces",
  "calories": 234.0,
  "protein": 18.0,
  "carbs": 1.8,
  "fat": 15.9,
  "scale_factor": 3.0,
  "status": "found",
  "note": "Estimated nutritional values; actual values can vary by brand, preparation method, and serving size."
}


In [20]:
# ── Direct tool call: lookup_calories for unknown food ────────────────────
print("=" * 55)
print("TOOL CALL: lookup_calories('pizza') — not in DB")
print("=" * 55)
result = tools.lookup_calories("pizza")
print(json.dumps(result, indent=2))

TOOL CALL: lookup_calories('pizza') — not in DB
{
  "status": "not_found",
  "query": "pizza",
  "error": "'pizza' is not in the food database. Please try a different food name or check the supported list. Do not guess nutritional values for unknown foods."
}


In [21]:
# ── Direct tool call: check_snack_fit ────────────────────────────────────
print("=" * 55)
print("TOOL CALL: check_snack_fit('banana')")
print("=" * 55)
result = tools.check_snack_fit("banana")
print(json.dumps(result, indent=2))

TOOL CALL: check_snack_fit('banana')
{
  "status": "does_not_fit",
  "fits": false,
  "snack_food": "banana",
  "snack_calories": 105.0,
  "snack_protein": 1.3,
  "snack_carbs": 27.0,
  "snack_fat": 0.4,
  "quantity_str": "1 piece",
  "consumed_kcal": 306.0,
  "budget": 400.0,
  "remaining_kcal": 94.0,
  "surplus_deficit": -11.0,
  "decision": "\u274c NO \u2014 Banana (1 piece) is approximately 105.0 kcal. You only have 94.0 kcal remaining, so you would exceed your budget by 11.0 kcal.",
  "macro_preview": {
    "protein": 1.3,
    "carbs": 27.0,
    "fat": 0.4
  },
  "note": "Estimated nutritional values; actual values can vary by brand, preparation method, and serving size."
}


---
## 📊 Scenario 5 — Full Day Simulation

A realistic multi-turn conversation demonstrating memory across all turns.

In [22]:
# ── Full fresh session ───────────────────────────────────────────────────
tools.reset_session()
agent._turn = 0

conversations = [
    "My calorie budget is 2000 kcal",
    "Add 2 eggs",
    "Add 200 g chicken breast",
    "Can I have a 300 g soya chunk snack?",
    "Can I eat an apple instead?",
    "Add an apple",
    "Show today's summary",
]

print("🍽️  FULL DAY SIMULATION — 7-turn conversation")
print("   (Budget: 2000 kcal | Watch agent remember across all turns)")
print("─" * 60)

for user_msg in conversations:
    agent.chat(user_msg)
    print()   # extra blank line between turns

🍽️  FULL DAY SIMULATION — 7-turn conversation
   (Budget: 2000 kcal | Watch agent remember across all turns)
────────────────────────────────────────────────────────────

════════════════════════════════════════════════════════════
  👤 USER (Turn 1): My calorie budget is 2000 kcal
════════════════════════════════════════════════════════════
  [PLAN] Intent detected → SET_BUDGET

───────────────────────────────────────────────────────
  🤖 AGENT — Setting daily calorie budget
───────────────────────────────────────────────────────
  [ACT] Calling set_budget(2000.0)
  [TOOL RESULT] Status = set, budget = 2000.0 kcal

  🤖 RESPONSE:
✅ Daily calorie budget set to 2000 kcal.
   Start logging your meals and I'll track calories and macros for you.

⚠️  All nutritional values are estimated. Actual values vary by brand, preparation, and serving size. This tool is for informational purposes only.


════════════════════════════════════════════════════════════
  👤 USER (Turn 2): Add 2 eggs
═════════

---
## ✅ Summary: Agentic Requirements Checklist

| Requirement | Status | Evidence |
|---|---|---|
| Real agent (not simple chatbot) | ✅ | Plan-act loop in `agent.py` |
| Tool 1: `lookup_calories()` | ✅ | Called before every add/check |
| Tool 2: `add_meal()` | ✅ | Stores in `SessionMemory` |
| Memory across turns | ✅ | `SessionMemory` in `memory.py` |
| Tool results drive decisions | ✅ | `check_snack_fit` reads memory + tool result |
| Macro split (Group of 3) | ✅ | Protein/Carbs/Fat % in every add response |
| Multi-step trace visible | ✅ | `[PLAN][ACT][TOOL RESULT][OBSERVE][DECIDE]` logs |
| Unknown food handled | ✅ | `not_found` status, no invented values |
| Safety disclaimer | ✅ | Appended to every response |
| Remaining budget calculation | ✅ | `remaining_calories()` in memory |

---
## 🎓 Viva Q&A

**Q1: What makes this an *agent* and not a chatbot?**  
An agent decides *which tool to call* and *what to do next* based on the results. Here, the agent detects the intent, calls `lookup_calories()`, reads the kcal result, then decides whether to call `add_meal()` or `check_snack_fit()`, then reads memory, then composes its response. A chatbot would just generate text.

**Q2: How does memory work across turns?**  
`SessionMemory` (in `memory.py`) stores the budget, meals list, and running totals as Python data structures. Each tool call updates these values. The agent reads them via `get_memory()`. This is programmatic — not stored in the LLM's text context.

**Q3: What is the plan-act loop?**  
1. **Plan**: `detect_intent()` reads the user message and decides what to do.  
2. **Act**: The agent calls the appropriate tool(s).  
3. **Observe**: The agent reads the tool's return value.  
4. **Decide**: Based on the observation, the agent either calls another tool or responds.  
5. **Respond**: Compose the final answer using tool results + memory.

**Q4: What happens when a food is unknown?**  
`lookup_calories()` returns `{"status": "not_found"}`. The agent checks this and tells the user the food is unavailable. It never invents or guesses nutritional values.

**Q5: How is the macro split calculated?**  
Using standard dietary formulas:  
- Protein calories = protein_g × 4  
- Carb calories = carbs_g × 4  
- Fat calories = fat_g × 9  
Then each macro's share of the total macro calories is expressed as a percentage.

## agent.chat("My calorie budget is 2500kcal")


In [23]:
agent.chat("Can I eat an apple?")



════════════════════════════════════════════════════════════
  👤 USER (Turn 8): Can I eat an apple?
════════════════════════════════════════════════════════════
  [PLAN] Intent detected → CHECK_SNACK

───────────────────────────────────────────────────────
  🤖 AGENT — Checking snack feasibility
───────────────────────────────────────────────────────
  [MEMORY] Budget = 2000.0, Consumed = 581.0 kcal
  [ACT] Calling lookup_calories('apple')
  [TOOL RESULT] Status = found
  [OBSERVE] Snack = 'apple' → 95.0 kcal
  [ACT] Calling check_snack_fit('apple')
  [TOOL RESULT] Status = fits
  [OBSERVE] Remaining budget = 1419.0 kcal | Snack = 95.0 kcal | Surplus/deficit = 1324.0 kcal
  [DECIDE] FITS ✅

  🤖 RESPONSE:
✅ YES — Apple (1 piece) is approximately 95.0 kcal. You have 1419.0 kcal remaining, so it fits your budget with 1324.0 kcal to spare.

  Snack nutrition:
  🥩 Protein: 0.5 g | 🍞 Carbs: 25.0 g | 🧈 Fat: 0.3 g

  Budget overview:
  [█████░░░░░░░░░░░░░░░] 29.0%
  Consumed : 581.0 / 2000.0 k

'✅ YES — Apple (1 piece) is approximately 95.0 kcal. You have 1419.0 kcal remaining, so it fits your budget with 1324.0 kcal to spare.\n\n  Snack nutrition:\n  🥩 Protein: 0.5 g | 🍞 Carbs: 25.0 g | 🧈 Fat: 0.3 g\n\n  Budget overview:\n  [█████░░░░░░░░░░░░░░░] 29.0%\n  Consumed : 581.0 / 2000.0 kcal\n  Remaining: 1419.0 kcal ✅\n\n⚠️  All nutritional values are estimated. Actual values vary by brand, preparation, and serving size. This tool is for informational purposes only.'

In [24]:
tools.reset_session()
agent._turn = 0
agent.chat("My calorie budget is 2500kcal")



════════════════════════════════════════════════════════════
  👤 USER (Turn 1): My calorie budget is 2500kcal
════════════════════════════════════════════════════════════
  [PLAN] Intent detected → SET_BUDGET

───────────────────────────────────────────────────────
  🤖 AGENT — Setting daily calorie budget
───────────────────────────────────────────────────────
  [ACT] Calling set_budget(2500.0)
  [TOOL RESULT] Status = set, budget = 2500.0 kcal

  🤖 RESPONSE:
✅ Daily calorie budget set to 2500 kcal.
   Start logging your meals and I'll track calories and macros for you.

⚠️  All nutritional values are estimated. Actual values vary by brand, preparation, and serving size. This tool is for informational purposes only.


"✅ Daily calorie budget set to 2500 kcal.\n   Start logging your meals and I'll track calories and macros for you.\n\n⚠️  All nutritional values are estimated. Actual values vary by brand, preparation, and serving size. This tool is for informational purposes only."

In [25]:
agent.chat("Add palak")


════════════════════════════════════════════════════════════
  👤 USER (Turn 2): Add palak
════════════════════════════════════════════════════════════
  [PLAN] Intent detected → ADD_MEAL

───────────────────────────────────────────────────────
  🤖 AGENT — Logging a meal
───────────────────────────────────────────────────────
  [ACT] Calling lookup_calories('palak')
  [TOOL RESULT] Status = found
  [OBSERVE] Found 'spinach' — 0.2 kcal, P:0.0g C:0.0g F:0.0g
  [ACT] Calling add_meal('palak')
  [TOOL RESULT] Status = added, Daily total = 0.2 kcal
  [MEMORY] Consumed = 0.2 / 2500.0 kcal | Meals logged = 1

  🤖 RESPONSE:
✅ Added: Spinach (1 piece)
  🔥 Calories: 0.2 kcal
  🥩 Protein : 0.0 g
  🍞 Carbs   : 0.0 g
  🧈 Fat     : 0.0 g
  📅 Today: 0.2 / 2500.0 kcal
  ⏳ Remaining: 2499.8 kcal

  (No macros logged yet.)

⚠️  All nutritional values are estimated. Actual values vary by brand, preparation, and serving size. This tool is for informational purposes only.


'✅ Added: Spinach (1 piece)\n  🔥 Calories: 0.2 kcal\n  🥩 Protein : 0.0 g\n  🍞 Carbs   : 0.0 g\n  🧈 Fat     : 0.0 g\n  📅 Today: 0.2 / 2500.0 kcal\n  ⏳ Remaining: 2499.8 kcal\n\n  (No macros logged yet.)\n\n⚠️  All nutritional values are estimated. Actual values vary by brand, preparation, and serving size. This tool is for informational purposes only.'

In [26]:
agent.chat("Add dahi") 


════════════════════════════════════════════════════════════
  👤 USER (Turn 3): Add dahi
════════════════════════════════════════════════════════════
  [PLAN] Intent detected → ADD_MEAL

───────────────────────────────────────────────────────
  🤖 AGENT — Logging a meal
───────────────────────────────────────────────────────
  [ACT] Calling lookup_calories('dahi')
  [TOOL RESULT] Status = found
  [OBSERVE] Found 'curd' — 1.0 kcal, P:0.1g C:0.0g F:0.0g
  [ACT] Calling add_meal('dahi')
  [TOOL RESULT] Status = added, Daily total = 1.2 kcal
  [MEMORY] Consumed = 1.2 / 2500.0 kcal | Meals logged = 2

  🤖 RESPONSE:
✅ Added: Curd (1 piece)
  🔥 Calories: 1.0 kcal
  🥩 Protein : 0.1 g
  🍞 Carbs   : 0.0 g
  🧈 Fat     : 0.0 g
  📅 Today: 1.2 / 2500.0 kcal
  ⏳ Remaining: 2498.8 kcal

  📊 Macro Split (approximate):
     Protein : 0.1 g  → 100.0% of macro calories
     Carbs   : 0.0 g  → 0.0% of macro calories
     Fat     : 0.0 g  → 0.0% of macro calories
  ⚡ Total from macros: ~0.4 kcal
  (Protein×

'✅ Added: Curd (1 piece)\n  🔥 Calories: 1.0 kcal\n  🥩 Protein : 0.1 g\n  🍞 Carbs   : 0.0 g\n  🧈 Fat     : 0.0 g\n  📅 Today: 1.2 / 2500.0 kcal\n  ⏳ Remaining: 2498.8 kcal\n\n  📊 Macro Split (approximate):\n     Protein : 0.1 g  → 100.0% of macro calories\n     Carbs   : 0.0 g  → 0.0% of macro calories\n     Fat     : 0.0 g  → 0.0% of macro calories\n  ⚡ Total from macros: ~0.4 kcal\n  (Protein×4 + Carbs×4 + Fat×9 — approximate; for informational use only)\n\n⚠️  All nutritional values are estimated. Actual values vary by brand, preparation, and serving size. This tool is for informational purposes only.'

In [27]:
agent.chat("Add 3 eggs, 250ml milk, 40g oats, 100gm paneer")


════════════════════════════════════════════════════════════
  👤 USER (Turn 4): Add 3 eggs, 250ml milk, 40g oats, 100gm paneer
════════════════════════════════════════════════════════════
  [PLAN] Intent detected → ADD_MEAL
  [PLAN] Multi-food input detected — 4 items to log

───────────────────────────────────────────────────────
  🤖 AGENT — Logging 4 foods
───────────────────────────────────────────────────────
  [PLAN] Will call lookup_calories + add_meal for each: ['3 eggs', '250ml milk', '40g oats', '100gm paneer']
  [ACT] Calling lookup_calories('3 eggs')
  [TOOL RESULT] Status = found
  [OBSERVE] Found 'egg' — 234.0 kcal P:18.0g C:1.8g F:15.9g
  [ACT] Calling add_meal('3 eggs')
  [TOOL RESULT] Status = added, Running total = 235.2 kcal
  [ACT] Calling lookup_calories('250ml milk')
  [TOOL RESULT] Status = found
  [OBSERVE] Found 'milk' — 155.2 kcal P:8.3g C:12.5g F:8.3g
  [ACT] Calling add_meal('250ml milk')
  [TOOL RESULT] Status = added, Running total = 390.4 kcal
  [ACT] Cal

'✅ Added 4 food(s):\n\n  • Egg (3 pieces) — 234.0 kcal | P:18.0g C:1.8g F:15.9g\n  • Milk (250.0 ml) — 155.2 kcal | P:8.3g C:12.5g F:8.3g\n  • Oats (40.0 g) — 148.0 kcal | P:5.4g C:26.6g F:2.6g\n  • Paneer (100.0 g) — 265.0 kcal | P:18.3g C:1.2g F:20.8g\n\n  📅 Today: 803.4 / 2500.0 kcal\n  ⏳ Remaining: 1696.6 kcal\n\n  📊 Macro Split (approximate):\n     Protein : 50.1 g  → 25.1% of macro calories\n     Carbs   : 42.1 g  → 21.1% of macro calories\n     Fat     : 47.6 g  → 53.7% of macro calories\n  ⚡ Total from macros: ~797.2 kcal\n  (Protein×4 + Carbs×4 + Fat×9 — approximate; for informational use only)\n\n⚠️  All nutritional values are estimated. Actual values vary by brand, preparation, and serving size. This tool is for informational purposes only.'

In [28]:
show summary

SyntaxError: invalid syntax (2605658662.py, line 1)

In [29]:
agent.chat("Show today's summary")


════════════════════════════════════════════════════════════
  👤 USER (Turn 5): Show today's summary
════════════════════════════════════════════════════════════
  [PLAN] Intent detected → SHOW_SUMMARY

───────────────────────────────────────────────────────
  🤖 AGENT — Retrieving daily summary
───────────────────────────────────────────────────────
  [ACT] Calling get_daily_summary()
  [TOOL RESULT] Budget=2500.0, Consumed=803.4 kcal, Meals=6

  🤖 RESPONSE:
📋 Today's Meal Log:
   1. Spinach (1 piece) — 0.2 kcal | P:0.0g C:0.0g F:0.0g
   2. Curd (1 piece) — 1.0 kcal | P:0.1g C:0.0g F:0.0g
   3. Egg (3 pieces) — 234.0 kcal | P:18.0g C:1.8g F:15.9g
   4. Milk (250.0 ml) — 155.2 kcal | P:8.3g C:12.5g F:8.3g
   5. Oats (40.0 g) — 148.0 kcal | P:5.4g C:26.6g F:2.6g
   6. Paneer (100.0 g) — 265.0 kcal | P:18.3g C:1.2g F:20.8g

  [██████░░░░░░░░░░░░░░] 32.1%
  Consumed : 803.4 / 2500.0 kcal
  Remaining: 1696.6 kcal ✅

  📊 Macro Split (approximate):
     Protein : 50.1 g  → 25.1% of macro cal

"📋 Today's Meal Log:\n   1. Spinach (1 piece) — 0.2 kcal | P:0.0g C:0.0g F:0.0g\n   2. Curd (1 piece) — 1.0 kcal | P:0.1g C:0.0g F:0.0g\n   3. Egg (3 pieces) — 234.0 kcal | P:18.0g C:1.8g F:15.9g\n   4. Milk (250.0 ml) — 155.2 kcal | P:8.3g C:12.5g F:8.3g\n   5. Oats (40.0 g) — 148.0 kcal | P:5.4g C:26.6g F:2.6g\n   6. Paneer (100.0 g) — 265.0 kcal | P:18.3g C:1.2g F:20.8g\n\n  [██████░░░░░░░░░░░░░░] 32.1%\n  Consumed : 803.4 / 2500.0 kcal\n  Remaining: 1696.6 kcal ✅\n\n  📊 Macro Split (approximate):\n     Protein : 50.1 g  → 25.1% of macro calories\n     Carbs   : 42.1 g  → 21.1% of macro calories\n     Fat     : 47.6 g  → 53.7% of macro calories\n  ⚡ Total from macros: ~797.2 kcal\n  (Protein×4 + Carbs×4 + Fat×9 — approximate; for informational use only)\n\n⚠️  All nutritional values are estimated. Actual values vary by brand, preparation, and serving size. This tool is for informational purposes only."

In [30]:
agent.chat("How many calories have I consumed?")



════════════════════════════════════════════════════════════
  👤 USER (Turn 6): How many calories have I consumed?
════════════════════════════════════════════════════════════
  [PLAN] Intent detected → SHOW_SUMMARY

───────────────────────────────────────────────────────
  🤖 AGENT — Retrieving daily summary
───────────────────────────────────────────────────────
  [ACT] Calling get_daily_summary()
  [TOOL RESULT] Budget=2500.0, Consumed=803.4 kcal, Meals=6

  🤖 RESPONSE:
📋 Today's Meal Log:
   1. Spinach (1 piece) — 0.2 kcal | P:0.0g C:0.0g F:0.0g
   2. Curd (1 piece) — 1.0 kcal | P:0.1g C:0.0g F:0.0g
   3. Egg (3 pieces) — 234.0 kcal | P:18.0g C:1.8g F:15.9g
   4. Milk (250.0 ml) — 155.2 kcal | P:8.3g C:12.5g F:8.3g
   5. Oats (40.0 g) — 148.0 kcal | P:5.4g C:26.6g F:2.6g
   6. Paneer (100.0 g) — 265.0 kcal | P:18.3g C:1.2g F:20.8g

  [██████░░░░░░░░░░░░░░] 32.1%
  Consumed : 803.4 / 2500.0 kcal
  Remaining: 1696.6 kcal ✅

  📊 Macro Split (approximate):
     Protein : 50.1 g  → 25.1

"📋 Today's Meal Log:\n   1. Spinach (1 piece) — 0.2 kcal | P:0.0g C:0.0g F:0.0g\n   2. Curd (1 piece) — 1.0 kcal | P:0.1g C:0.0g F:0.0g\n   3. Egg (3 pieces) — 234.0 kcal | P:18.0g C:1.8g F:15.9g\n   4. Milk (250.0 ml) — 155.2 kcal | P:8.3g C:12.5g F:8.3g\n   5. Oats (40.0 g) — 148.0 kcal | P:5.4g C:26.6g F:2.6g\n   6. Paneer (100.0 g) — 265.0 kcal | P:18.3g C:1.2g F:20.8g\n\n  [██████░░░░░░░░░░░░░░] 32.1%\n  Consumed : 803.4 / 2500.0 kcal\n  Remaining: 1696.6 kcal ✅\n\n  📊 Macro Split (approximate):\n     Protein : 50.1 g  → 25.1% of macro calories\n     Carbs   : 42.1 g  → 21.1% of macro calories\n     Fat     : 47.6 g  → 53.7% of macro calories\n  ⚡ Total from macros: ~797.2 kcal\n  (Protein×4 + Carbs×4 + Fat×9 — approximate; for informational use only)\n\n⚠️  All nutritional values are estimated. Actual values vary by brand, preparation, and serving size. This tool is for informational purposes only."

In [31]:
import importlib
import food_database, memory, tools, agent as agent_module
importlib.reload(food_database)
importlib.reload(memory)
importlib.reload(tools)
importlib.reload(agent_module)
from agent import MealCaloricAgent

tools.reset_session()
agent = MealCaloricAgent(verbose=True)
print("Reloaded with fix ✅")

Reloaded with fix ✅
